# Notebook 6: Timecourse Analysis

## Purpose
Test **H6**: Emotional arc over story time (begin→end).

## Analysis
- **Repeated Measures ANOVA** (or mixed effects) with:
  - Within factor: `segment (begin, middle, end)`
  - Dependent vars: `commitment`, `repair`, `miscommunication`, `neg_affect`
- **Trend plots** of arcs for Top vs Trash

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

np.random.seed(42)

## 1. Load Segment-Level Data

In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent

# Try to load chapter-level indices if available
CHAPTER_INDICES = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "indices_chapter.csv"
BOOK_INDICES = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "indices_book.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis"

if CHAPTER_INDICES.exists():
    df_segments = pd.read_csv(CHAPTER_INDICES)
    print(f"✓ Loaded chapter-level data: {len(df_segments)} rows")
    print(f"  Books: {df_segments['book_id'].nunique()}")
    print(f"  Segments: {df_segments['segment'].unique() if 'segment' in df_segments.columns else 'N/A'}")
else:
    print(f"⚠ Chapter-level indices not found. Need to compute from sentence-level data.")
    print(f"  This requires segmenting books into begin/middle/end and computing indices per segment.")
    df_segments = None

## 2. Prepare Data for Repeated Measures ANOVA

In [ ]:
# Define dependent variables for H6
dependent_vars = ['commitment', 'repair', 'miscommunication', 'neg_affect']

if df_segments is not None and 'segment' in df_segments.columns:
    # Ensure segment is categorical with proper order
    df_segments['segment'] = pd.Categorical(
        df_segments['segment'], 
        categories=['begin', 'middle', 'end'], 
        ordered=True
    )
    
    # Filter to available dependent variables
    available_vars = [var for var in dependent_vars if var in df_segments.columns]
    print(f"Available dependent variables: {available_vars}")
    
    # Check data structure
    print(f"\nData structure:")
    print(df_segments.groupby(['book_id', 'segment']).size().head(10))

## 3. Repeated Measures ANOVA

In [ ]:
def repeated_measures_anova(data, subject='book_id', within='segment', dv='commitment'):
    """Perform repeated measures ANOVA."""
    # Reshape data for AnovaRM
    pivot_data = data.pivot_table(
        index=subject,
        columns=within,
        values=dv
    ).dropna()
    
    if len(pivot_data) < 3:
        print(f"Insufficient data for {dv}")
        return None
    
    # Reshape to long format for AnovaRM
    long_data = pivot_data.reset_index().melt(
        id_vars=subject,
        value_name=dv,
        var_name=within
    )
    
    try:
        anova = AnovaRM(long_data, dv=dv, subject=subject, within=[within])
        result = anova.fit()
        return result
    except Exception as e:
        print(f"Error in ANOVA for {dv}: {e}")
        return None

# Run repeated measures ANOVA for each dependent variable
anova_results = []
if df_segments is not None and 'segment' in df_segments.columns:
    for dv in available_vars:
        result = repeated_measures_anova(df_segments, dv=dv)
        if result is not None:
            anova_results.append({
                'dependent_variable': dv,
                'f_statistic': result.anova_table.loc['segment', 'F Value'],
                'p_value': result.anova_table.loc['segment', 'Pr > F'],
                'df_num': result.anova_table.loc['segment', 'DF Num'],
                'df_den': result.anova_table.loc['segment', 'DF Den']
            })
            print(f"\n{dv}:")
            print(result.anova_table)
    
    anova_df = pd.DataFrame(anova_results)
    print("\n\nSummary of Repeated Measures ANOVA:")
    print(anova_df)

## 4. Trend Plots: Emotional Arcs by Group

In [ ]:
# Plot timecourse for each dependent variable, separated by group
if df_segments is not None and 'segment' in df_segments.columns:
    if 'group' in df_segments.columns:
        for dv in available_vars:
            plt.figure(figsize=(10, 6))
            
            # Compute means and standard errors per group and segment
            summary = df_segments.groupby(['group', 'segment'])[dv].agg(['mean', 'sem']).reset_index()
            
            for group in summary['group'].unique():
                group_data = summary[summary['group'] == group]
                x_pos = [0, 1, 2]  # begin, middle, end
                plt.errorbar(
                    x_pos,
                    group_data['mean'],
                    yerr=group_data['sem'],
                    marker='o',
                    label=group,
                    capsize=5
                )
            
            plt.xticks(x_pos, ['Begin', 'Middle', 'End'])
            plt.xlabel('Story Segment')
            plt.ylabel(f'{dv.capitalize()}')
            plt.title(f'Emotional Arc: {dv.capitalize()} by Group (H6)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / f'timecourse_{dv}_by_group.png')
            plt.show()
    else:
        # Plot without group separation
        for dv in available_vars:
            plt.figure(figsize=(10, 6))
            summary = df_segments.groupby('segment')[dv].agg(['mean', 'sem']).reset_index()
            x_pos = [0, 1, 2]
            plt.errorbar(
                x_pos,
                summary['mean'],
                yerr=summary['sem'],
                marker='o',
                capsize=5
            )
            plt.xticks(x_pos, ['Begin', 'Middle', 'End'])
            plt.xlabel('Story Segment')
            plt.ylabel(f'{dv.capitalize()}')
            plt.title(f'Emotional Arc: {dv.capitalize()} (H6)')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(OUTPUT_DIR / f'timecourse_{dv}.png')
            plt.show()

## 5. Post-hoc Tests for Segment Differences

In [ ]:
# Pairwise comparisons between segments (begin vs middle, middle vs end, begin vs end)
if df_segments is not None and 'segment' in df_segments.columns:
    posthoc_results = []
    
    for dv in available_vars:
        segments = ['begin', 'middle', 'end']
        for i, seg1 in enumerate(segments):
            for seg2 in segments[i+1:]:
                data1 = df_segments[df_segments['segment'] == seg1][dv].dropna()
                data2 = df_segments[df_segments['segment'] == seg2][dv].dropna()
                
                if len(data1) > 0 and len(data2) > 0:
                    # Paired t-test (if same books across segments)
                    # Or Mann-Whitney U if not paired
                    stat, p_val = stats.wilcoxon(data1, data2) if len(data1) == len(data2) else stats.mannwhitneyu(data1, data2)
                    
                    posthoc_results.append({
                        'dependent_variable': dv,
                        'segment1': seg1,
                        'segment2': seg2,
                        'p_value': p_val,
                        'mean_diff': data1.mean() - data2.mean()
                    })
    
    if posthoc_results:
        posthoc_df = pd.DataFrame(posthoc_results)
        
        # Apply multiple comparison correction
        p_values = posthoc_df['p_value'].values
        _, p_corrected, _, _ = multipletests(p_values, method='holm')
        posthoc_df['p_value_corrected'] = p_corrected
        
        print("Post-hoc Pairwise Comparisons:")
        print(posthoc_df)
        
        # Save
        output_file = OUTPUT_DIR / "timecourse_posthoc_results.csv"
        posthoc_df.to_csv(output_file, index=False)
        print(f"\n✓ Saved: {output_file}")

## 6. Save Results

In [ ]:
# Save ANOVA results
if 'anova_df' in locals():
    output_file = OUTPUT_DIR / "timecourse_anova_results.csv"
    anova_df.to_csv(output_file, index=False)
    print(f"✓ Saved: {output_file}")

## Summary

Timecourse analysis complete. H6 tested. Next: Notebook 7 (Robustness & Sensitivity)